In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
import matplotlib.pyplot as plt
from utils.preprocess_images import DeepfakeDataset, IMAGE_TRANSFORM_TRAIN, IMAGE_TRANSFORM_VAL, get_images_paths_faces
from utils.visualizations import view_picture
import torchvision.models as models
import torch.nn as nn
from torch.utils.data import DataLoader
import torch

In [ ]:
images_paths_train, images_paths_val, labels_train, labels_val  = get_images_paths_faces('/kaggle/input/datasets/PRANABR0Y/CelebDF_V2(image Dataset)')

In [ ]:
view_picture(images_paths_val[67])

In [ ]:
dataset_train = DeepfakeDataset(images_paths_train, labels_train, transform=IMAGE_TRANSFORM_TRAIN)
dataset_test = DeepfakeDataset(images_paths_val, labels_val, transform=IMAGE_TRANSFORM_VAL)

train_loader = DataLoader(
    dataset_train ,
    batch_size=32,
    shuffle=True,
    num_workers=6) 

val_loader = DataLoader(
    dataset_test,
    batch_size=32,
    shuffle = False,
    num_workers=2)

len(dataset_train), len(dataset_test)

In [ ]:
def train_model_with_early_stopping(num_epochs:int = 20):  

    criterion = nn.CrossEntropyLoss()
    #settings for early stopping
    best_val_loss = float('inf') 
    patience = 3                
    epochs_no_improve = 0         
    save_path = "best_resnet_fc.pth"

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}")
        resnet.train()
        train_loss = 0
        train_batches = 0

        for images, labels in train_loader:
            images = images.to(device_pc)
            labels = labels.to(device_pc)

            optimizer.zero_grad()
            outputs = resnet(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_batches += 1
        
        avg_train_loss = train_loss/train_batches
        print(f"Train loss: {avg_train_loss:.4f}")

        resnet.eval()
        val_loss = 0
        correct = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device_pc)
                labels = labels.to(device_pc)

                outputs = resnet(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)
                correct += (preds == labels).sum().item()
        

        avg_val_loss = val_loss/len(val_loader.dataset)
        accuracy = correct / len(val_loader.dataset)
        scheduler.step()

        print(f"Val loss: {avg_val_loss:.4f}")
        print(f"Val acc: {accuracy:.4f}")

        # EARLY STOPPING
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            
            torch.save(resnet.state_dict(), save_path)
            print(f"Model saved at epoch {epoch+1}")
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print('There is no improvement')
            break

In [ ]:
device_pc = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

resnet = models.resnet18(pretrained=True)

for param in resnet.parameters():
    param.requires_grad = True

resnet.fc = nn.Linear(resnet.fc.in_features, 2)
optimizer = torch.optim.Adam(resnet.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
resnet = resnet.to(device_pc)
print("Device:", device_pc)

In [ ]:
train_model_with_early_stopping(10)

In [ ]:
!find /kaggle -name "*.pth"